# 07 Walk Forward Out of Sample Backtest

Run the full daily strategy with frozen formation parameters, lagged next-close instructions, exact forecast horizons, available-cash funding and a maximum of ten open positions. This is the only notebook that runs the main portfolio simulation.

Run cells from top to bottom, or choose **Run All** for this notebook only. Each module saves its outputs for the next notebook. Restart the kernel after pulling code changes.


## 1. Imports and saved run


In [ ]:
%matplotlib inline
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'research_config.py').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Open Jupyter inside the Pairs_trading repository.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.project_io import OUTPUT_DIR, initialize, load_config, load_frame, save_frame, save_json

cfg = load_config()


## 2. Load frozen formation data and settings

Module 04 and Module 06 previews are not fed in as a replacement for daily forecasting. The engine uses their shared functions across the complete test period.


In [ ]:
from scripts.run_research import read_series, backtest_kwargs
from src.backtest import run_walk_forward_backtest
from src.research_validation import portfolio_metrics
train_prices = load_frame('train_prices')
test_prices = load_frame('test_prices')
eligible_pairs = load_frame('eligible_pairs')
cointegrated_pairs = load_frame('cointegrated_pairs')
risk_free_rates = read_series(OUTPUT_DIR / 'inputs/rates.parquet')
display(pd.Series(backtest_kwargs(cfg)))


## 3. Run the simulation

This cell may take time. Diagnostics in Modules 08–11 load the saved result and do not rerun it.


In [ ]:
result = run_walk_forward_backtest(train_prices, test_prices, eligible_pairs, cointegrated_pairs,
                                  risk_free_rates, **backtest_kwargs(cfg))
trades = result['trades']
equity_curve = result['equity_curve']
summary = portfolio_metrics(result, cfg.initial_capital)
for name, frame in result.items():
    save_frame(name, frame)
save_json('backtest_summary', summary)
display(pd.Series(summary, name='Backtest results'))


## 4. Accounting and timing checks

These checks fail if funding, timing or cash reconciliation differs from the declared methodology.


In [ ]:
assert equity_curve.cash.min() >= -1e-7
assert cfg.max_open_pairs is None or equity_curve.n_open_positions.max() <= cfg.max_open_pairs
assert np.allclose(equity_curve.equity, equity_curve.cash + equity_curve.open_position_value)
assert np.isclose(equity_curve.equity.iloc[-1], cfg.initial_capital + trades.pnl.sum())
if not trades.empty:
    assert (trades.entry_date > trades.signal_date).all()
    assert (trades.entry_premium <= trades.entry_budget + 1e-7).all()
display(trades.head(10))
equity_curve.equity.plot(figsize=(11, 4), title='Synthetic option portfolio')
plt.ylabel('Model equity')
plt.show()


## Save module completion

Wait for this confirmation before moving to the next notebook.


In [ ]:
print(f'Completed. Files saved in {OUTPUT_DIR}')
